In [ ]:
import os
import tarfile

def compress_parquet_folder(input_folder, output_file):
    """
    Compress all parquet files in a folder into a tar.gz archive.
    """
    with tarfile.open(output_file, "w:gz") as tar:
        for root, dirs, files in os.walk(input_folder):
            for file in files:
                if file.endswith(".parquet"):
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, input_folder)
                    
                    print(f"Adding: {arcname}")
                    tar.add(full_path, arcname=arcname)

    print(f"\nDone! Archive created: {output_file}")



compress_parquet_folder("trades", "trades.tar.gz")
compress_parquet_folder("open_interest", "open_interest.tar.gz")
compress_parquet_folder("liquidations", "liquidations.tar.gz")
compress_parquet_folder("orderbook", "trades.tar.gz")

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "cryptohftdata"])

In [ ]:
import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc  # <--- Important for memory management

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-08-01"
END_DATE = "2025-08-30" 
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES

DROP_HEADERS = ['some_unused_column', 'internal_id', 'order_count', 'transaction_time', 'event_time', 'timestamp', 'first_update_id', 'final_update_id', 'prev_final_update_id', 'last_update_id'] 

client = chd.CryptoHFTDataClient(api_key=API_KEY)

def optimize_floats(df):
    """Downcast floats to save 50% memory."""
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df

def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(delta.days + 1)]

def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)
    categories = [
        ("trades", client.get_trades),
        ("open_interest", client.get_open_interest),
        ("liquidations", client.get_liquidations),
        ("orderbook", client.get_orderbook)
    ]

    for cat_name, fetch_func in categories:
        print(f"\n>>> Starting Category: {cat_name.upper()}")
        os.makedirs(cat_name, exist_ok=True)

        for date_str in dates:
            file_path = f"{cat_name}/{date_str}_{SYMBOL}.parquet"

            if os.path.exists(file_path):
                print(f"  [SKIP] {date_str} already exists.")
                continue

            try:
                print(f"  [FETCH] {date_str}...", end="\r")
                df = fetch_func(
                    symbol=SYMBOL,
                    exchange=EXCHANGE,
                    start_date=date_str,
                    end_date=date_str
                )

                if df is not None and not df.empty:
                    # 1. Drop unused columns immediately
                    df.drop(columns=[c for c in DROP_HEADERS if c in df.columns], errors='ignore', inplace=True)

                    # 2. Convert time
                    if 'received_time' in df.columns:
                        df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

                    # 3. Optimize Memory usage (Shrink floats)
                    df = optimize_floats(df)

                    # 4. Save to Disk
                    df.to_parquet(file_path, engine='pyarrow', compression='zstd', compression_level=9, index=False)
                    
                    print(f"  [SAVED] {date_str} | Rows: {len(df):,}")
                    
                    # 5. AGGRESSIVE MEMORY CLEANUP
                    del df
                    gc.collect() # Manually trigger garbage collection
                else:
                    print(f"  [EMPTY] {date_str} - No data found.")

            except Exception as e:
                print(f"\n  [ERROR] {date_str}: {e}")
                # Clean up even on error to prevent leaks
                if 'df' in locals(): del df
                gc.collect()

if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")

In [ ]:
import polars as pl
import pandas as pd
from sortedcontainers import SortedDict
from pathlib import Path
import pickle
import os

def reconstruct_and_featurize(
    filepath: str,
    output_dir: str,
    state_path: str,          # Pfad zum State-File (pickle)
    resample_interval: str = "30s",
):
    # --- State laden (vom Vortag) oder leer initialisieren ---
    if os.path.exists(state_path):
        with open(state_path, "rb") as f:
            bids, asks = pickle.load(f)
        print(f"  State geladen: {len(bids)} bid-levels, {len(asks)} ask-levels")
    else:
        bids = SortedDict(lambda x: -x)
        asks = SortedDict()
        print("  Kein State gefunden, starte leer")

    # --- Einlesen & auf 1s aggregieren ---
    df = pl.read_parquet(filepath)
    df = (
        df
        .with_columns(
            pl.col("received_time").dt.truncate("1s").alias("ts_1s")
        )
        .sort(["ts_1s", "side", "price", "received_time"])
        .group_by(["ts_1s", "side", "price"])
        .agg(pl.col("quantity").last())
        .sort("ts_1s")
    )

    # --- Lückenlose Zeitachse für den Tag erzeugen ---
    # Damit leere Sekunden später mit ffill gefüllt werden können
    day_str = Path(filepath).stem[:10]  # erwartet Dateiname wie "2025-08-01_..."
    day_start = pd.Timestamp(day_str)
    day_end   = day_start + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    full_index = pd.date_range(day_start, day_end, freq="1s")

    # --- Rekonstruktions-Loop ---
    records     = df.iter_rows(named=True)
    snapshots   = []
    current_ts  = None
    current_grp = []

    def process_group(ts, group):
        for row in group:
            book = bids if row["side"] == "bid" else asks
            if row["quantity"] == 0:
                book.pop(row["price"], None)
            else:
                book[row["price"]] = row["quantity"]

        if not bids or not asks:
            return

        N          = 10
        bid_prices = list(bids.keys())[:N]
        ask_prices = list(asks.keys())[:N]
        best_bid   = bid_prices[0]
        best_ask   = ask_prices[0]
        bid_vol_5  = sum(bids[p] for p in bid_prices[:5])
        ask_vol_5  = sum(asks[p] for p in ask_prices[:5])
        bid_vol_10 = sum(bids[p] for p in bid_prices)
        ask_vol_10 = sum(asks[p] for p in ask_prices)

        snapshots.append({
            "time":       ts,
            "mid_price":  (best_bid + best_ask) / 2,
            "spread":     best_ask - best_bid,
            "spread_bps": (best_ask - best_bid) / best_bid * 10000,
            "obi_5":      (bid_vol_5  - ask_vol_5)  / (bid_vol_5  + ask_vol_5),
            "obi_10":     (bid_vol_10 - ask_vol_10) / (bid_vol_10 + ask_vol_10),
            "bid_vol_5":  bid_vol_5,
            "ask_vol_5":  ask_vol_5,
        })

    for row in records:
        ts = row["ts_1s"]
        if ts != current_ts:
            if current_grp:
                process_group(current_ts, current_grp)
            current_ts  = ts
            current_grp = [row]
        else:
            current_grp.append(row)
    if current_grp:
        process_group(current_ts, current_grp)

    # --- State für nächsten Tag speichern ---
    with open(state_path, "wb") as f:
        pickle.dump((bids, asks), f)
    print(f"  State gespeichert: {len(bids)} bid-levels, {len(asks)} ask-levels")

    # --- Leere Sekunden auffüllen ---
    result = (
        pd.DataFrame(snapshots)
        .set_index("time")
        .reindex(full_index)          # alle 86400 Sekunden erzwingen
        .ffill()                      # leere Sekunden mit letztem bekannten Wert füllen
        .dropna()                     # Anfang des ersten Tags falls noch kein State
    )

    # --- Auf gewünschtes Intervall resamplen ---
    result = (
        result
        .resample(resample_interval)
        .agg({
            "mid_price":  ["first", "last", "mean"],
            "spread_bps": ["mean", "max"],
            "obi_5":      ["mean", "std", "last"],
            "obi_10":     ["mean", "last"],
            "bid_vol_5":  "mean",
            "ask_vol_5":  "mean",
        })
    )
    result.columns = ["_".join(c) for c in result.columns]

    out_path = Path(output_dir) / (Path(filepath).stem + "_features.parquet")
    result.to_parquet(out_path)
    print(f"  → {len(result)} Buckets gespeichert")


def process_all(input_dir: str, output_dir: str, state_path: str = "ob_state.pkl"):
    files = sorted(Path(input_dir).glob("*.parquet"))  # sort → chronologische Reihenfolge
    os.makedirs(output_dir, exist_ok=True)

    for f in files:
        print(f"Verarbeite: {f.name}")
        reconstruct_and_featurize(str(f), output_dir, state_path)


if __name__ == "__main__":
    process_all(
        input_dir  = "/pfad/zu/orderbook/",
        output_dir = "/pfad/zu/features/",
        state_path = "/pfad/zu/ob_state.pkl",
    )